# Infosys Springboard Internship 7.0 — Milestone 1
## User Authentication Module (Login · Signup · Forgot Password) — Streamlit + JWT + ngrok + Gmail OTP

**What this notebook does**
- Builds a single Streamlit app with **User** and **Admin** login tabs
- Signup with Username, Email, Password, Confirm Password, Security Question & Answer
- Forgot Password with **two routes**: Security Question *or* Email OTP (Gmail SMTP)
- JWT-based session handling (2-hour expiry for users)
- Field validation: mandatory fields, email format, strong password rule
- User Dashboard (welcome + logout) and separate Admin Dashboard (lists all registered users, never passwords)
- Publicly exposed via **ngrok**

**Before running:** make sure these Colab Secrets are set (key icon in the left sidebar), each with notebook access enabled:

| Secret | Purpose |
|---|---|
| `JWT_SECRET` | Random string used to sign JWT session tokens |
| `NGROK_AUTHTOKEN` | Your ngrok Authtoken |
| `EMAIL_ADDRESS` | Gmail address that sends OTP emails |
| `EMAIL_PASSWORD` | Gmail App Password (not your normal Gmail password) |
| `ADMIN_USERNAME` *(optional)* | Defaults to `admin` if not set |
| `ADMIN_PASSWORD` *(optional)* | Defaults to `Admin@123` if not set — **set this secret for a real deployment** |

Run the three cells below in order.

### Step 1 — Install dependencies

In [ ]:
!pip install -q streamlit pyjwt bcrypt pyngrok pandas

### Step 2 — Write the Streamlit app (`app.py`)

In [ ]:
%%writefile app.py
import os, re, sqlite3, jwt, bcrypt, datetime, time, secrets, smtplib, streamlit as st
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.utils import formatdate, make_msgid

# ============================================================
# 0. SECRETS  (Colab Secrets -> environment, with local fallback)
# ============================================================
def _secret(name, default=None):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name, default)

JWT_SECRET      = _secret("JWT_SECRET", "dev-only-fallback-secret")
EMAIL_ADDRESS   = _secret("EMAIL_ADDRESS")
EMAIL_PASSWORD  = _secret("EMAIL_PASSWORD")
ADMIN_USERNAME  = _secret("ADMIN_USERNAME", "admin")
ADMIN_PASSWORD  = _secret("ADMIN_PASSWORD", "Admin@123")   # override via Colab Secret ADMIN_PASSWORD
OTP_EXPIRY_MIN  = 5

# ============================================================
# 1. PAGE CONFIG + THEME
# ============================================================
st.set_page_config(page_title="Infosys Auth Portal", page_icon="🔐", layout="wide", initial_sidebar_state="expanded")

os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#6C63FF"\nbackgroundColor="#F5F6FB"\n'
            'secondaryBackgroundColor="#FFFFFF"\ntextColor="#1E1B4B"\nfont="sans serif"\n')

C = {
    "bg1": "#EEF1FF", "bg2": "#E4E9FF",
    "card": "#FFFFFF", "card_border": "rgba(108,99,255,0.15)",
    "text": "#1E1B4B", "muted": "#6B7280",
    "primary": "#6C63FF", "primary_dark": "#4C46B6",
    "accent": "#00C9A7", "danger": "#FF4D6D", "warn": "#FFB020",
    "admin": "#1E1B4B",
}

st.markdown(f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@500;600;700;800&family=Inter:wght@400;500;600&display=swap');

html, body, .stApp {{
    background: radial-gradient(circle at top left, {C['bg2']} 0%, {C['bg1']} 45%, #ffffff 100%) !important;
    font-family: 'Inter', sans-serif !important;
    color: {C['text']} !important;
}}
#MainMenu, footer, header {{ visibility: hidden !important; }}
div[data-testid="stDecoration"] {{ display: none !important; }}
.block-container {{ padding: 2rem 3rem 3rem !important; max-width: 1180px; }}
h1, h2, h3, h4 {{ font-family: 'Poppins', sans-serif !important; color: {C['text']} !important; letter-spacing: -0.02em; }}
label p {{ font-weight: 600 !important; color: {C['text']} !important; font-size: 13.5px !important; }}

/* Glass auth card */
.auth-card {{
    background: {C['card']};
    border: 1px solid {C['card_border']};
    border-radius: 22px;
    padding: 40px 44px 32px;
    box-shadow: 0 20px 45px -12px rgba(108,99,255,0.28), 0 0 0 1px rgba(255,255,255,0.6) inset;
}}
.brand-badge {{
    width: 62px; height: 62px; border-radius: 18px; display:flex; align-items:center; justify-content:center;
    background: linear-gradient(135deg, {C['primary']}, {C['accent']}); font-size:28px; margin: 0 auto 14px;
    box-shadow: 0 10px 25px -6px rgba(108,99,255,0.55);
}}
.pill {{
    display:inline-block; padding:4px 14px; border-radius:30px; font-size:11.5px; font-weight:700;
    letter-spacing:0.04em; text-transform:uppercase; background:rgba(108,99,255,0.12); color:{C['primary_dark']};
}}

/* Inputs */
div[data-baseweb="base-input"], div[data-baseweb="select"] > div {{ background-color: transparent !important; border:none !important; }}
div[data-baseweb="input"], div[data-baseweb="select"] {{
    background-color: #F8F9FF !important; border: 1.5px solid #E3E6F5 !important; border-radius: 12px !important;
}}
div[data-baseweb="input"]:focus-within, div[data-baseweb="select"]:focus-within {{
    border-color: {C['primary']} !important; box-shadow: 0 0 0 4px rgba(108,99,255,0.12) !important;
}}
input, div[data-baseweb="select"] span {{ color: {C['text']} !important; -webkit-text-fill-color: {C['text']} !important; font-size: 14.5px !important; }}
input::placeholder {{ color: #A0A4C0 !important; }}

/* Buttons */
div[data-testid="stButton"] button {{
    background: linear-gradient(135deg, {C['primary']}, {C['primary_dark']}) !important;
    color: #ffffff !important; border: none !important; border-radius: 12px !important;
    font-family:'Inter',sans-serif !important; font-weight:700 !important; font-size:14.5px !important;
    height: 46px !important; width:100%; box-shadow: 0 10px 20px -8px rgba(108,99,255,0.55) !important;
    transition: all 0.18s ease !important;
}}
div[data-testid="stButton"] button:hover {{ transform: translateY(-2px); box-shadow: 0 14px 26px -8px rgba(108,99,255,0.65) !important; }}
div[data-testid="stButton"] button p {{ color:#fff !important; }}

/* Secondary (ghost) buttons via help text trick not needed - use container styling per-column below */
.ghost-btn button {{
    background: #F1F1FB !important; color: {C['primary_dark']} !important;
    box-shadow: none !important; border: 1.5px solid #E3E6F5 !important;
}}
.ghost-btn button p {{ color: {C['primary_dark']} !important; }}

section[data-testid="stSidebar"] {{ background: #ffffff !important; border-right: 1px solid #ECEEF9 !important; }}
div[data-testid="stVerticalBlockBorderWrapper"] {{
    background: {C['card']} !important; border: 1px solid {C['card_border']} !important;
    border-radius: 18px !important; box-shadow: 0 12px 30px -14px rgba(30,27,75,0.18) !important;
}}
.stTabs [data-baseweb="tab-list"] {{ gap: 4px; }}
.stTabs [data-baseweb="tab"] {{ border-radius: 10px 10px 0 0; font-weight:600; }}
hr {{ border-color: #ECEEF9 !important; }}
.stAlert {{ border-radius: 12px !important; }}
</style>
""", unsafe_allow_html=True)

# ============================================================
# 2. DATABASE
# ============================================================
DB_NAME = "infosys_auth.db"

def get_db():
    return sqlite3.connect(DB_NAME, check_same_thread=False)

def init_db():
    with get_db() as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            email TEXT UNIQUE NOT NULL,
            password_hash TEXT NOT NULL,
            security_question TEXT NOT NULL,
            security_answer_hash TEXT NOT NULL,
            created_at TEXT NOT NULL
        )""")

init_db()

def hash_txt(t: str) -> str:
    return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()

def check_txt(t: str, h: str) -> bool:
    try:
        return bcrypt.checkpw(t.encode(), h.encode()) if h else False
    except Exception:
        return False

def now_ts():
    return datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

# ============================================================
# 3. VALIDATION
# ============================================================
EMAIL_RE = re.compile(r"^[A-Za-z]{2,}[A-Za-z0-9._%+\-]*@[A-Za-z0-9]{2,}(\.[A-Za-z0-9]+)*\.[A-Za-z]{2,}$")
PASSWORD_RE = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[!@#$%^&*()_+\-=\[\]{};':\"\\|,.<>\/?~`]).{8,}$")

def is_valid_email(email: str) -> bool:
    return bool(EMAIL_RE.match(email or ""))

def is_valid_password(pwd: str) -> bool:
    return bool(PASSWORD_RE.match(pwd or ""))

PASSWORD_HELP = "Min 8 chars, 1 uppercase, 1 lowercase, 1 number & 1 special symbol."

# ============================================================
# 4. JWT SESSION HELPERS
# ============================================================
def make_jwt(subject: str, role: str = "user", minutes: int = 120):
    payload = {"sub": subject, "role": role,
               "iat": datetime.datetime.utcnow(),
               "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=minutes)}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_jwt(token: str):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception:
        return None

# ============================================================
# 5. OTP + EMAIL HELPERS
# ============================================================
def generate_otp() -> str:
    return f"{secrets.randbelow(900000) + 100000}"

def make_otp_token(email: str, otp: str):
    payload = {"sub": email, "otp_hash": hash_txt(otp), "type": "password_reset_otp",
               "iat": datetime.datetime.utcnow(),
               "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MIN)}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_otp_token(token: str, input_otp: str, email: str):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        if payload.get("sub") != email or payload.get("type") != "password_reset_otp":
            return False, "Security token mismatch."
        if check_txt(input_otp, payload["otp_hash"]):
            return True, "Valid"
        return False, "Invalid 6-digit code."
    except jwt.ExpiredSignatureError:
        return False, f"This OTP expired after {OTP_EXPIRY_MIN} minutes. Please request a new one."
    except Exception:
        return False, "Invalid or corrupted verification token."

def send_otp_email(to_email: str, otp: str):
    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
        return False, "Email sender not configured. Add EMAIL_ADDRESS & EMAIL_PASSWORD to Colab Secrets."
    msg = MIMEMultipart("alternative")
    msg["From"] = f"Infosys Auth Portal <{EMAIL_ADDRESS}>"
    msg["To"] = to_email
    msg["Subject"] = "Infosys Auth Portal - Your Verification Code"
    msg["Date"] = formatdate(localtime=True)
    msg["Message-ID"] = make_msgid()

    text_body = (f"Your verification code is: {otp}\n"
                 f"This code expires in {OTP_EXPIRY_MIN} minutes.\n"
                 f"If you did not request this, please ignore this email.")
    html_body = f"""
    <html><body style="font-family:Arial,sans-serif;background:#F5F6FB;padding:24px;">
      <div style="max-width:460px;margin:auto;background:#fff;border-radius:16px;padding:32px;text-align:center;border:1px solid #E3E6F5;">
        <div style="font-size:15px;color:#6B7280;margin-bottom:6px;">INFOSYS AUTH PORTAL</div>
        <h2 style="color:#1E1B4B;margin:0 0 14px;">Verify your identity</h2>
        <p style="color:#4B5563;font-size:14px;">Use the code below to continue with <b>{to_email}</b>.</p>
        <div style="font-size:30px;font-weight:800;letter-spacing:8px;background:#EEF1FF;color:#4C46B6;
                    padding:14px 10px;border-radius:12px;margin:18px 0;">{otp}</div>
        <p style="color:#9CA3AF;font-size:12px;">Expires in {OTP_EXPIRY_MIN} minutes. Ignore if you didn't request this.</p>
      </div>
    </body></html>"""
    msg.attach(MIMEText(text_body, "plain"))
    msg.attach(MIMEText(html_body, "html"))
    try:
        s = smtplib.SMTP("smtp.gmail.com", 587)
        s.starttls()
        s.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        s.sendmail(EMAIL_ADDRESS, to_email, msg.as_string())
        s.quit()
        return True, "OTP sent."
    except Exception as e:
        return False, f"SMTP error: {e}"

# ============================================================
# 6. SESSION STATE
# ============================================================
DEFAULTS = {
    "token": None, "role": None, "page": "Login",
    "forgot_mode": None,          # "sq" or "otp"
    "forgot_stage": "start",      # start -> verify -> reset
    "forgot_username": None, "forgot_email": None,
    "sq_text": None, "otp_token": None,
}
for k, v in DEFAULTS.items():
    if k not in st.session_state:
        st.session_state[k] = v

def goto(page: str):
    st.session_state.page = page
    st.rerun()

def reset_forgot_state():
    for k in ["forgot_mode", "forgot_stage", "forgot_username", "forgot_email", "sq_text", "otp_token"]:
        st.session_state[k] = DEFAULTS[k]

def brand_header(title: str, subtitle: str):
    st.markdown(f"""
    <div style="text-align:center;">
        <div class="brand-badge">🔐</div>
        <h1 style="font-size:1.7rem !important;margin:0;">Infosys Auth Portal</h1>
        <p style="color:{C['muted']};font-size:13.5px;margin:4px 0 18px;">{subtitle}</p>
        <span class="pill">{title}</span>
    </div>
    <div style="height:18px;"></div>
    """, unsafe_allow_html=True)

# ============================================================
# 7. AUTH PAGES  (Login / Signup / Forgot Password / Admin Login)
# ============================================================
def render_auth():
    _, mid, _ = st.columns([1, 1.3, 1])
    with mid:
        st.markdown('<div class="auth-card">', unsafe_allow_html=True)

        top_tabs = st.tabs(["👤 User", "🛡️ Admin"])

        # ---------------- USER TABS ----------------
        with top_tabs[0]:
            if st.session_state.page == "Admin":
                st.session_state.page = "Login"

            sub_tabs = st.tabs(["Sign In", "Create Account", "Forgot Password"])

            # ---- LOGIN ----
            with sub_tabs[0]:
                brand_header("Sign In", "Welcome back — enter your details")
                identifier = st.text_input("Username or Email", placeholder="jane_doe or jane@infosys.com", key="li_id")
                pwd = st.text_input("Password", type="password", placeholder="••••••••", key="li_pw")
                st.markdown("<div style='height:8px;'></div>", unsafe_allow_html=True)
                if st.button("Sign In →", key="btn_login"):
                    if not identifier or not pwd:
                        st.error("⚠️ Please fill in all fields.")
                    else:
                        ident = identifier.strip().lower()
                        with get_db() as c:
                            row = c.execute(
                                "SELECT username, email, password_hash FROM users WHERE lower(username)=? OR lower(email)=?",
                                (ident, ident)).fetchone()
                        if row and check_txt(pwd, row[2]):
                            st.session_state.token = make_jwt(row[1], role="user")
                            st.session_state.role = "user"
                            st.success("✅ Login successful! Redirecting…")
                            time.sleep(0.6)
                            st.rerun()
                        else:
                            st.error("❌ Invalid username/email or password.")

            # ---- SIGNUP ----
            with sub_tabs[1]:
                brand_header("Create Account", "Join the Infosys Auth Portal")
                uname = st.text_input("Username", placeholder="jane_doe", key="su_un")
                email = st.text_input("Email address", placeholder="jane@infosys.com", key="su_em")
                c1, c2 = st.columns(2)
                pwd = c1.text_input("Password", type="password", placeholder="••••••••", key="su_pw", help=PASSWORD_HELP)
                cpwd = c2.text_input("Confirm Password", type="password", placeholder="••••••••", key="su_cpw")
                sq = st.selectbox("Security Question", [
                    "What is your pet's name?",
                    "What is your mother's maiden name?",
                    "What city were you born in?",
                    "What is your favourite teacher's name?",
                ], key="su_sq")
                sa = st.text_input("Security Answer", placeholder="Your answer", key="su_sa")
                st.markdown("<div style='height:8px;'></div>", unsafe_allow_html=True)

                if st.button("Create Account →", key="btn_signup"):
                    if not all([uname, email, pwd, cpwd, sa]):
                        st.error("⚠️ Please fill in all fields — nothing can be left empty.")
                    elif not is_valid_email(email):
                        st.error("❌ Enter a valid email, e.g. ab@cd.ef")
                    elif not is_valid_password(pwd):
                        st.error(f"❌ Weak password. {PASSWORD_HELP}")
                    elif pwd != cpwd:
                        st.error("❌ Password and Confirm Password do not match.")
                    else:
                        try:
                            with get_db() as c:
                                c.execute(
                                    "INSERT INTO users (username, email, password_hash, security_question, security_answer_hash, created_at) VALUES (?,?,?,?,?,?)",
                                    (uname.strip(), email.strip().lower(), hash_txt(pwd), sq, hash_txt(sa.strip().lower()), now_ts()))
                            st.success("✅ Account created! Please sign in.")
                            time.sleep(1)
                            st.rerun()
                        except sqlite3.IntegrityError:
                            st.error("❌ That username or email is already registered.")

            # ---- FORGOT PASSWORD ----
            with sub_tabs[2]:
                brand_header("Forgot Password", "Recover access to your account")

                if st.session_state.forgot_stage == "start":
                    st.write("Choose a recovery method:")
                    method_tabs = st.tabs(["🔑 Security Question", "📧 Email OTP"])

                    # --- Security Question route ---
                    with method_tabs[0]:
                        f_uname = st.text_input("Username", placeholder="jane_doe", key="fp_sq_uname")
                        if st.button("Continue →", key="fp_sq_continue"):
                            with get_db() as c:
                                row = c.execute("SELECT security_question FROM users WHERE lower(username)=?",
                                                (f_uname.strip().lower(),)).fetchone()
                            if row:
                                st.session_state.forgot_mode = "sq"
                                st.session_state.forgot_stage = "verify"
                                st.session_state.forgot_username = f_uname.strip()
                                st.session_state.sq_text = row[0]
                                st.rerun()
                            else:
                                st.error("❌ Username not found.")

                    # --- OTP route ---
                    with method_tabs[1]:
                        f_email = st.text_input("Registered email address", placeholder="jane@infosys.com", key="fp_otp_email")
                        if st.button("Send OTP →", key="fp_otp_send"):
                            if not is_valid_email(f_email):
                                st.error("❌ Enter a valid registered email.")
                            else:
                                with get_db() as c:
                                    row = c.execute("SELECT 1 FROM users WHERE lower(email)=?",
                                                    (f_email.strip().lower(),)).fetchone()
                                if not row:
                                    st.error("❌ Email not registered.")
                                else:
                                    otp = generate_otp()
                                    with st.spinner("Sending OTP…"):
                                        ok, msg = send_otp_email(f_email.strip().lower(), otp)
                                    if ok:
                                        st.session_state.forgot_mode = "otp"
                                        st.session_state.forgot_stage = "verify"
                                        st.session_state.forgot_email = f_email.strip().lower()
                                        st.session_state.otp_token = make_otp_token(f_email.strip().lower(), otp)
                                        st.success("✅ OTP sent! Check your inbox.")
                                        time.sleep(0.8)
                                        st.rerun()
                                    else:
                                        st.error(f"❌ {msg}")

                elif st.session_state.forgot_stage == "verify":
                    if st.session_state.forgot_mode == "sq":
                        st.info(f"❓ **{st.session_state.sq_text}**")
                        ans = st.text_input("Your Answer", key="fp_sq_answer")
                        if st.button("Verify Answer →", key="fp_sq_verify"):
                            with get_db() as c:
                                row = c.execute("SELECT security_answer_hash FROM users WHERE lower(username)=?",
                                                (st.session_state.forgot_username.lower(),)).fetchone()
                            if row and check_txt(ans.strip().lower(), row[0]):
                                st.session_state.forgot_stage = "reset"
                                st.rerun()
                            else:
                                st.error("❌ Incorrect answer.")
                    else:
                        st.info(f"📧 Code sent to **{st.session_state.forgot_email}** (valid {OTP_EXPIRY_MIN} min).")
                        otp_in = st.text_input("6-digit OTP", max_chars=6, key="fp_otp_input")
                        if st.button("Verify OTP →", key="fp_otp_verify"):
                            ok, msg = verify_otp_token(st.session_state.otp_token, otp_in.strip(), st.session_state.forgot_email)
                            if ok:
                                st.session_state.forgot_stage = "reset"
                                st.rerun()
                            else:
                                st.error(f"❌ {msg}")
                    if st.button("← Cancel", key="fp_verify_cancel"):
                        reset_forgot_state(); st.rerun()

                elif st.session_state.forgot_stage == "reset":
                    st.success("✅ Identity verified — set a new password.")
                    npw = st.text_input("New Password", type="password", key="fp_new_pw", help=PASSWORD_HELP)
                    cnpw = st.text_input("Confirm New Password", type="password", key="fp_confirm_pw")
                    if st.button("Update Password →", key="fp_update"):
                        if not is_valid_password(npw):
                            st.error(f"❌ Weak password. {PASSWORD_HELP}")
                        elif npw != cnpw:
                            st.error("❌ Passwords do not match.")
                        else:
                            if st.session_state.forgot_mode == "sq":
                                with get_db() as c:
                                    c.execute("UPDATE users SET password_hash=? WHERE lower(username)=?",
                                              (hash_txt(npw), st.session_state.forgot_username.lower()))
                            else:
                                with get_db() as c:
                                    c.execute("UPDATE users SET password_hash=? WHERE lower(email)=?",
                                              (hash_txt(npw), st.session_state.forgot_email))
                            st.success("🎉 Password updated! Please sign in.")
                            time.sleep(1)
                            reset_forgot_state()
                            st.rerun()
                    if st.button("← Cancel", key="fp_reset_cancel"):
                        reset_forgot_state(); st.rerun()

        # ---------------- ADMIN TAB ----------------
        with top_tabs[1]:
            brand_header("Admin Sign In", "Restricted access — administrators only")
            a_user = st.text_input("Admin Username", placeholder="admin", key="admin_user")
            a_pass = st.text_input("Admin Password", type="password", placeholder="••••••••", key="admin_pass")
            st.markdown("<div style='height:8px;'></div>", unsafe_allow_html=True)
            if st.button("Sign In as Admin →", key="btn_admin_login"):
                if a_user == ADMIN_USERNAME and a_pass == ADMIN_PASSWORD:
                    st.session_state.token = make_jwt(ADMIN_USERNAME, role="admin")
                    st.session_state.role = "admin"
                    st.success("✅ Admin verified! Redirecting…")
                    time.sleep(0.6)
                    st.rerun()
                else:
                    st.error("❌ Invalid admin credentials.")

        st.markdown('</div>', unsafe_allow_html=True)

# ============================================================
# 8. DASHBOARDS
# ============================================================
def top_bar(title: str, subtitle: str, badge_text: str):
    st.markdown(f"""
    <div style="background:{C['admin']};border-radius:18px;padding:22px 30px;display:flex;
                justify-content:space-between;align-items:center;margin-bottom:26px;">
        <div>
            <div style="color:{C['accent']};font-weight:800;font-size:20px;">🔐 {title}</div>
            <div style="color:#C7CBEF;font-size:12.5px;">{subtitle}</div>
        </div>
        <div style="background:{C['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{C['admin']};font-size:13px;">
            {badge_text}
        </div>
    </div>""", unsafe_allow_html=True)

def render_user_dashboard(email: str):
    with get_db() as c:
        row = c.execute("SELECT username, created_at FROM users WHERE email=?", (email,)).fetchone()
    uname, created = row if row else (email, "-")

    with st.sidebar:
        st.markdown(f"""<div style="text-align:center;padding:18px 6px;">
            <div style="font-size:30px;">🔐</div>
            <div style="font-weight:700;font-size:15px;">Infosys Auth Portal</div>
            <div style="font-size:11.5px;color:{C['muted']};">User Dashboard</div>
        </div><hr>""", unsafe_allow_html=True)
        st.markdown(f"👋 Signed in as **{uname}**")
        if st.button("🚪 Logout", key="user_logout"):
            st.session_state.token = None
            st.session_state.role = None
            st.rerun()

    top_bar("Infosys Auth Portal", "Member Dashboard", f"👤 {uname}")

    st.markdown(f"""
    <div class="auth-card" style="text-align:center;padding:50px 20px;">
        <h2 style="margin-bottom:6px;">Welcome, {uname}! 🎉</h2>
        <p style="color:{C['muted']};font-size:14.5px;">You're securely signed in as <b>{email}</b> using a JWT session token.</p>
    </div>""", unsafe_allow_html=True)

    st.markdown("<div style='height:22px;'></div>", unsafe_allow_html=True)
    c1, c2, c3 = st.columns(3)
    for col, icon, label, value in [
        (c1, "🗓️", "Account Created", created.split(" ")[0] if created != "-" else "-"),
        (c2, "🛡️", "Auth Method", "JWT + bcrypt"),
        (c3, "✅", "Account Status", "Active"),
    ]:
        col.markdown(f"""
        <div class="auth-card" style="text-align:center;padding:22px 10px;">
            <div style="font-size:24px;">{icon}</div>
            <div style="font-size:17px;font-weight:700;margin-top:6px;">{value}</div>
            <div style="color:{C['muted']};font-size:12px;">{label}</div>
        </div>""", unsafe_allow_html=True)

def render_admin_dashboard():
    with st.sidebar:
        st.markdown(f"""<div style="text-align:center;padding:18px 6px;">
            <div style="font-size:30px;">🛡️</div>
            <div style="font-weight:700;font-size:15px;">Infosys Auth Portal</div>
            <div style="font-size:11.5px;color:{C['muted']};">Admin Panel</div>
        </div><hr>""", unsafe_allow_html=True)
        st.markdown(f"🛡️ Signed in as **{ADMIN_USERNAME}**")
        if st.button("🚪 Logout", key="admin_logout"):
            st.session_state.token = None
            st.session_state.role = None
            st.rerun()

    top_bar("Admin Control Panel", "All registered users", "🛡️ Administrator")

    with get_db() as c:
        rows = c.execute("SELECT username, email, created_at FROM users ORDER BY created_at DESC").fetchall()

    c1, c2 = st.columns(2)
    c1.markdown(f"""<div class="auth-card" style="text-align:center;">
        <div style="font-size:26px;">👥</div>
        <div style="font-size:24px;font-weight:800;">{len(rows)}</div>
        <div style="color:{C['muted']};font-size:12px;">Total Registered Users</div></div>""", unsafe_allow_html=True)
    c2.markdown(f"""<div class="auth-card" style="text-align:center;">
        <div style="font-size:26px;">🔒</div>
        <div style="font-size:24px;font-weight:800;">Secured</div>
        <div style="color:{C['muted']};font-size:12px;">Passwords stored as bcrypt hashes</div></div>""", unsafe_allow_html=True)

    st.markdown("<div style='height:20px;'></div>", unsafe_allow_html=True)
    st.markdown("#### 📋 Registered Users")
    if rows:
        import pandas as pd
        df = pd.DataFrame(rows, columns=["Username", "Email", "Joined"])
        st.dataframe(df, use_container_width=True, hide_index=True)
    else:
        st.info("No users have signed up yet.")

# ============================================================
# 9. ROUTER
# ============================================================
payload = verify_jwt(st.session_state.token) if st.session_state.token else None

if payload:
    if payload.get("role") == "admin":
        render_admin_dashboard()
    else:
        render_user_dashboard(payload["sub"])
else:
    st.session_state.token = None
    render_auth()


### Step 3 — Launch Streamlit + ngrok tunnel

In [ ]:
import os, time, subprocess
from pyngrok import ngrok
from google.colab import userdata

# 1. Load secrets from Colab Secrets into environment variables
for key in ["JWT_SECRET", "NGROK_AUTHTOKEN", "EMAIL_ADDRESS", "EMAIL_PASSWORD", "ADMIN_USERNAME", "ADMIN_PASSWORD"]:
    try:
        val = userdata.get(key)
        if val:
            os.environ[key] = val
    except Exception:
        pass

ngrok_token = os.environ.get("NGROK_AUTHTOKEN")
if not ngrok_token:
    raise RuntimeError("NGROK_AUTHTOKEN not found. Add it in Colab Secrets (key icon in the left sidebar).")
ngrok.set_auth_token(ngrok_token)

# 2. Kill any previous Streamlit / ngrok sessions
ngrok.kill()
get_ipython().system("pkill -f streamlit")
time.sleep(2)

# 3. Launch Streamlit in the background
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    env=os.environ.copy(),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)

# 4. Open the public ngrok tunnel
public_url = ngrok.connect(8501).public_url
print("=" * 60)
print(f"Infosys Auth Portal is LIVE at: {public_url}")
print("=" * 60)
print("Leave this cell running. Press the Colab Stop button or Ctrl+C to shut down.")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nShutting down...")
    ngrok.kill()
    process.terminate()
    get_ipython().system("pkill -f streamlit")
    print("Ngrok tunnel closed and Streamlit stopped.")
